# ABA Demo — single-pass Colab candidate → human review → website replay

Primary product: live visual assistance. This notebook is the **recorded-video development/export path**, not measured live performance. Five experimental visual indicators remain separate: orientation (not gaze/attention), body motion, out of seat, hand motion, posture change. Unsupported/uncertain values remain `null`; no diagnosis, behavior function, treatment or clinical accuracy claim. Fixed-camera CFR footage only.

**Privacy:** authorization must cover Google Colab and every visible person. This notebook does not upload recordings. The optional visual-context cell sends only 1-4 confirmed scene/crop JPEGs with timestamps and target boxes to OpenRouter; the full video is never sent. No per-video consent gate is implemented in this demo. Use VS Code **Upload to Colab** manually for code ZIP and authorized recording. Clear all outputs before sharing; preview frames are sensitive. No video or secret belongs in the code bundle.

Open this NEW notebook rather than the stale demo_colab or Clean editor tab. Run stages in order; do not Run All through human decisions. If you change configuration or miss a reselection, start a new session, never seek backwards or repair earlier identities. A restart means one new candidate pass, not audit/export duplicate passes.

## 1. Verified clean code load
From the project root, generate the code-only ZIP with `python scripts/prepare_demo_colab_bundle.py --output demo_colab_bundle.zip`; do not commit it. Use VS Code **Upload to Colab** to place it at `/content/demo_colab_bundle.zip`. The pinned digest below checks the exact thirteen-member ZIP generated from these sources. Loading clears the fixed extracted directory and cached package imports, not your recording or prior uniquely named runs.

In [ ]:
from pathlib import Path, PurePosixPath
import base64
import hashlib
import io
import importlib
import json
import os
import shutil
import subprocess
import sys
import zipfile
from collections import Counter

bundle_path = Path('/content/demo_colab_bundle.zip')
root = Path('/content/aba_demo_project')
EXPECTED_BUNDLE_SHA256 = '3096f9a9f961e1cc275cdd4bef25e75e1341a64ea8dabe3fd1722a170bbffa5d'
allowed = {
    'aba_demo/__init__.py',
    'aba_demo/engine.py',
    'aba_demo/export.py',
    'aba_demo/vision.py',
    'aba_demo/botsort_reid.yaml',
    'aba_demo/context_schema.py',
    'aba_demo/openrouter_context.py',
    'requirements-demo-vision.txt',
    'tests/test_demo_vision.py',
    'aba_demo/colab_workflow.py',
    'tests/test_colab_workflow.py',
    'tests/test_context_schema.py',
    'tests/test_openrouter_context.py',
}

if not bundle_path.is_file():
    raise FileNotFoundError(
        'Upload demo_colab_bundle.zip with VS Code > Upload to Colab, then rerun this cell.'
    )

actual_bundle_sha256 = hashlib.sha256(bundle_path.read_bytes()).hexdigest()
if actual_bundle_sha256 != EXPECTED_BUNDLE_SHA256:
    raise ValueError(
        f'Bundle SHA-256 mismatch: expected {EXPECTED_BUNDLE_SHA256}, got {actual_bundle_sha256}'
    )

if root.is_symlink() or (root.exists() and not root.is_dir()):
    root.unlink()
elif root.exists():
    shutil.rmtree(root)
root.mkdir(parents=True, exist_ok=False)

with zipfile.ZipFile(bundle_path) as archive:
    members = [PurePosixPath(item.filename).as_posix() for item in archive.infolist() if not item.is_dir()]
    if len(members) != len(set(members)):
        raise ValueError('The code bundle contains duplicate members')
    if set(members) != allowed:
        missing = sorted(allowed - set(members))
        extra = sorted(set(members) - allowed)
        raise ValueError(f'Unexpected code bundle contents; missing={missing}, extra={extra}')
    for relative in members:
        target = root / relative
        target.parent.mkdir(parents=True, exist_ok=True)
        target.write_bytes(archive.read(relative))

for module_name in list(sys.modules):
    if module_name == 'aba_demo' or module_name.startswith('aba_demo.'):
        del sys.modules[module_name]
importlib.invalidate_caches()

os.chdir(root)
sys.path[:] = [entry for entry in sys.path if entry != str(root)]
sys.path.insert(0, str(root))

print('Reviewed project source ready:', root)
print('Bundle SHA-256:', actual_bundle_sha256)
print('Bundle members:', sorted(allowed))


## 2. Install and lightweight checks
These tests use synthetic dependency boundaries, not accuracy measurements. Heavy model execution happens only on this runtime.

In [ ]:
def run_checked(command):
    result = subprocess.run(command, capture_output=True, text=True)
    print(result.stdout, end='')
    if result.stderr:
        print(result.stderr, end='', file=sys.stderr)
    if result.returncode:
        raise RuntimeError(f'Command failed: exit {result.returncode}')
run_checked([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-demo-vision.txt'])
run_checked([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'])
import torch
DEVICE = 0 if torch.cuda.is_available() else 'cpu'
print('Inference device:', DEVICE)
from aba_demo.colab_workflow import ReviewSession, show_evidence


## 3. Configure once and preview
Set a runtime-local video path and calibrate task/seat rectangles `[x1,y1,x2,y2]` on the normalized scene. Empty ROIs honestly leave those measurements unavailable; enter them before starting the candidate if all five measurements are needed. Thresholds are provisional demo settings.

**Candidate overlap policy is not human approval.** `human_review_required` below uses overlap threshold 1.0 only for the pending candidate: continuous physical contact can otherwise make strict selection impossible. You must inspect the full montage and audit before publishing the exact artifact. Set `strict` to retain overlap suspension at 0.2. Both policies retain target-loss and same-ID jump latches; neither reconnects different tracker IDs. A candidate may be wrong-person evidence even when coverage passes.


In [ ]:
VIDEO_PATH = '/content/session.mp4'
video_path = Path(VIDEO_PATH)
if video_path.parent != Path('/content') or not video_path.is_file():
    raise ValueError('Manually Upload to Colab and use an existing video directly under /content')
CANDIDATE_OVERLAP_POLICY = 'human_review_required'  # candidate-only, NOT approved continuity
CONFIG = {'weights': 'yolo11s-pose.pt', 'device': DEVICE, 'sample_hz': 5,
          'task_targets': [], 'seat_roi': None, 'identity_recovery_max_gap_s': 0.5,
          'tracker_profile': 'botsort_reid',
          'candidate_overlap_policy': CANDIDATE_OVERLAP_POLICY}
if 'session' in globals():
    session.close()
session = ReviewSession(video_path, CONFIG, '/content/aba_review_runs')
show_evidence([session.preview])
print('Source SHA-256:', session.source_sha256)
print('Run:', session.run_id)


## 4. Human initial selection
Inspect the first-frame boxes above. Enter a normalized point inside exactly one target and its displayed ID. No target or approval is prefilled. Selection checks the actual returned ID and binds all subsequent evidence to this recording. A mismatch invalidates the run.

In [ ]:
TARGET_CLICK = None  # e.g. [x, y] from YOUR displayed frame
EXPECTED_INITIAL_ID = None
INITIAL_TARGET_CONFIRMED = False
if TARGET_CLICK is None:
    raise ValueError('Enter the reviewed first-frame target click')
print(session.select(TARGET_CLICK, expected_target_id=EXPECTED_INITIAL_ID,
                     confirmed=INITIAL_TARGET_CONFIRMED))


## 5. Bounded early causal probe, using the SAME analyzer
Advance to a reviewed early time appropriate for the current recording, then inspect the dense montage and current frame. The actual decoded time is shown (nominal frame rounding). Tracker IDs are recording-local and no different-ID mapping is authorized automatically. Change the probe time only forward; additional review pauses reuse this cell, not a new full inference pass.


In [ ]:
RESELECTION_REVIEW_TIME = 3.5  # clue only; inspect the actual current frame
current = session.advance(RESELECTION_REVIEW_TIME)
show_evidence([session.evidence[i] for i in sorted(session.evidence)])
show_evidence([current])
print('Current decoded time:', current['time'], 'boxes:', current['boxes'])


## 6. Optional explicit current-frame reselection
If the original lock is still correct, leave `APPLY_RESELECTION=False`. If fragmented, confirm the same participant in the CURRENT frame, then fill all fields and set both booleans true. Nothing automatically selects a different tracker ID. The previous uncertain gap remains null. In strict mode an overlap refusal requires a later unambiguous frame or restart with the explicitly declared human-review-required candidate policy. Continuous-contact candidate mode does not waive final human continuity review.


In [ ]:
APPLY_RESELECTION = False
RESELECTION_CLICK = None
EXPECTED_RESELECTED_ID = None
RESELECTION_CONFIRMED = False
if APPLY_RESELECTION:
    if RESELECTION_CLICK is None:
        raise ValueError('Enter a point on the CURRENT reviewed target')
    print(session.select(RESELECTION_CLICK, expected_target_id=EXPECTED_RESELECTED_ID,
                         confirmed=RESELECTION_CONFIRMED))
else:
    print('No reselection applied; uncertainty remains latched when applicable.')


## 7. Finish ONE candidate and inspect the full causal audit
This continues from the current frame to EOF. It records every decoded frame in a causal identity audit, preserves five indicators at the configured sample rate, and shows a bounded early/uniform/tail montage with the ACTIVE selected ID and reselection markers. Only terminal unreadable tail ≤3 frames is tolerated; early/larger EOF fails. A montage is sparse human evidence, not proof of identity accuracy. Review the complete audit JSON as needed. Failed quality stays quarantined under its unique run directory, never `observations.json`.

Limits: 100,000 causal frames, 100,000 observations, a 20 MiB JSON and a four-hour run (including review pauses). Use a shorter authorized development clip if exceeded. Visual evidence is at most 34 retained frames (early + uniform + safe tail + initial/reselection frames; up to 12 reselections), displayed in batches of 12. Progress prints every 100 frames. The uncertainty fraction is a configurable development coverage gate, NOT a safety/accuracy score.

In [ ]:
MAX_UNCERTAIN_FRACTION = 0.10  # development quality gate, not clinical validation
report = session.finish(max_uncertain_fraction=MAX_UNCERTAIN_FRACTION,
                        progress=lambda done, total: print(f'Tracked {done}/{total} frames', flush=True))
show_evidence([session.evidence[i] for i in sorted(session.evidence)])
print(json.dumps(report, indent=2))
print('Full causal audit and exact candidate:', session.pending)
print('No new inference is needed for publication.')


## 8. Optional visual context (OpenRouter demo)
Running this cell sends only 1-4 confirmed full-scene JPEGs, matching target crops, timestamps, and per-frame target boxes to the configured OpenRouter model. The full video is never sent. The configured Gemini model is a paid, per-token route; leave `CONTEXT_WINDOWS = []` to make no provider request. Outputs are advisory visible-context enums only, not diagnosis, attention, intent, function, treatment, or recommendations. No per-video approval gate is implemented; running the cell is the invocation.


In [ ]:
ABA_OPENROUTER_VLM_MODEL = 'google/gemini-2.5-flash'
CONTEXT_MAX_TOKENS = 4096
CONTEXT_WINDOWS = []  # e.g. [{'segment_id': 'ctx-000001', 'frame_indices': [0, 5]}] from session.evidence
if CONTEXT_WINDOWS:
    from getpass import getpass
    from google.colab import userdata
    from aba_demo.openrouter_context import OpenRouterContextAdapter

    try:
        openrouter_api_key = userdata.get('OPENROUTER_API_KEY')
    except Exception as secret_error:
        print('Colab Secrets are unavailable from the VS Code runtime; use the masked prompt below.')
        openrouter_api_key = getpass('OpenRouter API key (masked, runtime only): ')
    if not openrouter_api_key:
        raise ValueError('OPENROUTER_API_KEY is required')
    try:
        context_report = session.build_context_candidate(
            CONTEXT_WINDOWS,
            adapter=OpenRouterContextAdapter(
                api_key=openrouter_api_key,
                model=ABA_OPENROUTER_VLM_MODEL,
                max_tokens=CONTEXT_MAX_TOKENS,
            ),
        )
        print('Context candidate:', context_report)
    finally:
        openrouter_api_key = None
else:
    print('No context windows selected; tracking-only publication remains available.')


## 8. Human review publishes the EXACT pending bytes
Review every displayed frame, selection events, complete causal audit and quality report. Enter the exact candidate digest (and context digest when a context candidate was built, published together via publish_with_context) printed above, a reviewer label, and explicit confirmation only when satisfied. Do not approve based only on a passing coverage number. Source or candidate changes invalidate publication. Publication atomically renames the reviewed pending file without rerunning YOLO; a separate human-review receipt binds the digest. No previous run is implicitly authorized.

For human-review-required overlap candidates, confirmation also means you reviewed target continuity during close contact; a passing coverage count is not evidence of correct identity.

In [ ]:
REVIEWED_CANDIDATE_SHA256 = ''  # paste the reviewed report digest, not an automatic assignment
REVIEWED_CONTEXT_SHA256 = ''  # paste the reviewed context digest only when context was built
REVIEWER = ''
HUMAN_REVIEW_CONFIRMED = False
if REVIEWED_CONTEXT_SHA256:
    published = session.publish_with_context(REVIEWED_CANDIDATE_SHA256, REVIEWED_CONTEXT_SHA256,
                                           confirmed=HUMAN_REVIEW_CONFIRMED, reviewer=REVIEWER)
else:
    published = session.publish(REVIEWED_CANDIDATE_SHA256,
                                confirmed=HUMAN_REVIEW_CONFIRMED, reviewer=REVIEWER)
print('Verified published replay artifact:', published)


## 9. Website handoff (manual, no widget transfers)
Use VS Code Colab runtime Contents **Download...** on the exact printed `observations.json` and optionally `human-review.json` / `review-report.json`. Retain the same original video locally. In the main website use recorded/precomputed replay with this JSON and its SHA-matching video; this is not a live inference benchmark. Do not select another run's output. Do not share the recording or saved notebook outputs with the source package.

If the gate fails, inspect the uncertainty reasons: fragmentation requires an explicit current-frame reselection; overlap is a different failure. Restart only if changed configuration/earlier decisions require a new candidate. This delivery does not claim that the supplied footage passes the chosen candidate policy and human review; only Colab execution and actual human review can establish that.